In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
import warnings

# Отключаем предупреждения
warnings.filterwarnings('ignore')

In [10]:
# Загружаем данные
df = pd.read_csv('cleaned_data.csv')

In [11]:
# Удаление выбросов для целевой переменной SI
Q1 = df['SI'].quantile(0.25)
Q3 = df['SI'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

# Оставляем только те строки, где SI не превышает верхнюю границу выбросов
df = df[df['SI'] <= upper_bound]
print(f"Размер данных после удаления выбросов SI: {df.shape}")

Размер данных после удаления выбросов SI: (815, 89)


In [12]:
# Убираем все целевые переменные и классы, чтобы модель не могла подсмотреть ответ
targets_to_drop = ['IC50, mM', 'CC50, mM', 'SI', 'IC50_class', 'CC50_class', 'SI_median_class', 'SI_8_class']
X = df.drop(columns=targets_to_drop, errors='ignore')
y = df['SI']

# Разбиваем данные выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
# Масштабируем данные
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
# Задаем словарь с моделями
models = {
    "Ridge (Линейная регрессия)": (Ridge(), {'alpha': [100.0, 200.0]}),
    "Random Forest (Случайный лес)": (RandomForestRegressor(random_state=42), {'n_estimators': [50, 100], 'max_depth': [None, 10]}),
    "Gradient Boosting (Градиентный бустинг)": (GradientBoostingRegressor(random_state=42), {'n_estimators': [100, 300, 500], 'learning_rate': [0.01, 0.05, 0.1]}),
    "SVR (Опорные векторы)": (SVR(), {'C': [50.0, 100.0]}),
    "KNN (Ближайшие соседи)": (KNeighborsRegressor(), {'n_neighbors': [10, 20]})
}

In [15]:
# Обучение и оценка моделей
results = {}

for name, (model, params) in models.items():
    grid = GridSearchCV(model, params, cv=3, scoring='r2', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)

    # Предсказание на лучшей модели
    best_model = grid.best_estimator_
    predictions = best_model.predict(X_test_scaled)

    # Расчет метрик
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)

    results[name] = {'R2': r2, 'MAE': mae, 'Best Params': grid.best_params_}

In [16]:
# Итоговые результаты
print("Итоговые результаты сравнения моделей (SI):\n")
for name, metrics in results.items():
    print(f"Модель: {name}")
    print(f"Лучшие параметры: {metrics['Best Params']}")
    print(f"Метрика R2: {metrics['R2']:.3f}")
    print(f"Ошибка MAE: {metrics['MAE']:.3f}\n")

Итоговые результаты сравнения моделей (SI):

Модель: Ridge (Линейная регрессия)
Лучшие параметры: {'alpha': 100.0}
Метрика R2: 0.140
Ошибка MAE: 6.100

Модель: Random Forest (Случайный лес)
Лучшие параметры: {'max_depth': 10, 'n_estimators': 100}
Метрика R2: 0.120
Ошибка MAE: 6.009

Модель: Gradient Boosting (Градиентный бустинг)
Лучшие параметры: {'learning_rate': 0.01, 'n_estimators': 300}
Метрика R2: 0.105
Ошибка MAE: 6.234

Модель: SVR (Опорные векторы)
Лучшие параметры: {'C': 50.0}
Метрика R2: 0.128
Ошибка MAE: 5.502

Модель: KNN (Ближайшие соседи)
Лучшие параметры: {'n_neighbors': 20}
Метрика R2: 0.062
Ошибка MAE: 6.530



**Вывод для SI.**

В ходе прогнозирования SI результаты оказались самыми слабыми. Формально лучший R2 показала линейная регрессия Ridge (R2 = 0.140, MAE = 6.1), а наименьшую ошибку SVR (MAE = 5.5, R2 = 0.128). Случайный лес и бустинг показали R2 около 0.11-0.12, а модель KNN справилась хуже всего (R2 = 0.062).

**Применимость.** Текущие регрессионные модели неприменимы для количественного прогнозирования SI. Крайне низкие значения R2 (<0.15) говорят о том, что алгоритмы практически не улавливают связь между дескрипторами и таргетом, так как SI — это сложное расчетное отношение (CC50 к IC50).

**Рекомендации по улучшению.** Следует отказаться от прямой регрессии для SI. Логичнее и точнее предсказывать значения IC50 и CC50 по отдельности, а затем высчитывать SI из них, либо сразу перевести задачу в классификацию, предсказывая уже созданные категории (например, `SI_8_class`).